# Tokenization

A walkthrough of Karpathy's *Let's build the GPT Tokenizer*, kept as the record of where `tokenizer/basic.py` comes from — every function in the package is first written and run here. Two sections marked **[added]** go beyond the lecture: a compression comparison across GPT-2 and GPT-4, and a check that GPT-2's shipped tokenizer is the same two objects we build.

From the lecture's opening:

> Tokenization is at the heart of much of the weirdness of LLMs. Do not brush it off.
>
> Why can't LLMs spell words? Tokenization. Why can't they reverse a string? Tokenization. Why are they worse at non-English languages? Tokenization. Why are they bad at simple arithmetic? Tokenization. Why did GPT-2 have more trouble than necessary coding in Python? Tokenization. Why does an LLM halt when it sees the string `<|endoftext|>`? Tokenization. What is that warning about trailing whitespace? Tokenization. Why does it break on `SolidGoldMagikarp`? Tokenization. Why prefer YAML over JSON with LLMs? Tokenization. Why is an LLM not actually end-to-end language modeling? Tokenization.

A good tokenization web app to keep open while reading: <https://tiktokenizer.vercel.app>. Paste these and watch how they split:

```
127 + 677 = 804
1275 + 6773 = 8041

Egg.
I have an Egg.
egg.
EGG.

만나서 반가워요. 저는 OpenAI에서 개발한 대규모 언어 모델인 ChatGPT입니다.

for i in range(1, 101):
    if i % 3 == 0 and i % 5 == 0:
        print("FizzBuzz")
```

## Part 1: from text to integers

Before any machine learning happens, we have to answer a plumbing question: what integer sequence represents this 
string? There are three candidate answers, and the lecture works through why the third one wins. 

Candidate 1: one integer per character (Unicode code point)
A Python str is a sequence of code points (integers in [0, 0x10FFFF]) assigned by the Unicode standard.
ord() reveals them. 

In [1]:
# A Python 'str' is a sequence of Unicode CODE POINTS, not bytes and not "characters"
# in any typographic sense. What you see rendered is the font's interpretation of
# those code points, which can hide a great detail of structure.
"안녕하세요 👋 (hello in Korean!)"

'안녕하세요 👋 (hello in Korean!)'

In [2]:
# ord(ch) -> the integer code point of a single character.
# Note the enormous dynamic range in one short string:
#   ASCII letters/punctuation  -> 32..126
#   Hangul syllables           -> ~44000..55000
#   waving hand emoji          -> 128075 (U+1F44B)
# 26 code points for this string.
[ord(x)for x in "안녕하세요 👋 (hello in Korean!)"]

[50504,
 45397,
 54616,
 49464,
 50836,
 32,
 128075,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 75,
 111,
 114,
 101,
 97,
 110,
 33,
 41]

Why this doesn't work as a vocabulary. Three problems, any one of which is disqualifying:

1) Size. ~150,000 code points are currently assigned, out of a codespace of 1,114,112. A 150k-row embedding table
is enormous, and most rows would be touched a handful of times in training (or never) so you'd ship a model full
of undertrained embeddings. 
2) Instability. The unicode standard is a living document; new code points land every year. A vocabulary pinned to it
is a vocabulary that drifts out from under your model. 
3) No compression. One integer per character means context length is spent at the character level. The transformer's 
attention is O(n**2) in sequence length, so this is expensive in the most literal sense.

### Candidate 2: one integer per byte (UTF-8)

Encode the string to bytes instead. Now the vocabulary is exactly 256: fixed universal, and it can represent every
string that exists.

In [3]:
# .encode("utf-8") -> a `bytes` object; list(...) exposes the raw byte values in 0..255.
# Compare against the code-point list above:
#   each Hangul syllable  -> 3 bytes
#   the emoji             -> 4 bytes
#   each ASCII character  -> 1 byte
# So 26 code points became 39 bytes. That inflation is the price of a 256-symbol vocabulary.
list("안녕하세요 👋 (hello in Korean!)".encode("utf-8"))

[236,
 149,
 136,
 235,
 133,
 149,
 237,
 149,
 152,
 236,
 132,
 184,
 236,
 154,
 148,
 32,
 240,
 159,
 145,
 139,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 75,
 111,
 114,
 101,
 97,
 110,
 33,
 41]

Why raw bytes alone still don't work: 
The vocabulary is beautiful (256 entries, never changes, covers everything), but the *sequences* are ruinous. 
English text roughly doubles in length versus a word-level scheme, and non-Latin scripts inflate 3-4x. 
Feeding bytes straight to a transformer means a tiny vocabulary and a punishing context length — 
the model spends its capacity learning to spell instead of learning to reason.

This byte inflation is exactly *why* LLMs are worse at non-English languages. Korean or Japanese text costs 
several times more tokens than the equivalent English, so it eats context faster, and (because tokenizer training 
sets skew English) it also gets *fewer* useful merges. Both effects compound.

### Candidate 3: bytes + learned merges = byte pair encoding

Keep the 256 byte tokens as an unbreakable foundation, then **learn** additional tokens by repeatedly fusing 
the most frequent adjacent pair. We get a tunable vocabulary size, guaranteed coverage of any input (there's always 
a byte-level fallback), and sequences a few times shorter. That's the rest of the notebook.

## Byte pair encoding, from scratch

The algorithm, in full:

1. Encode the training text to a list of byte ids (0..255).
2. Count every adjacent pair.
3. Take the most frequent pair, mint a new id for it, and replace all its
   occurrences with that id.
4. Repeat until the vocabulary is the size you want.

In [4]:
# A paragraph of real text to train on. Deliberately messy: fullwidth characters,
# emoji, curly quotes and em-dashes, so we can watch UTF-8 do its thing.
# text from https://www.reedbeta.com/blog/programmers-intro-to-unicode/
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
tokens = text.encode("utf-8") # raw bytes
tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 for convenience
print('---')
print(text)
print("length:", len(text)) # number of CODE POINTS
print('---')
print(tokens)
print("length:", len(tokens)) # number of BYTES -- strictly larger, and the gap is the fancy characters

---
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
length: 533
---
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140

### Count adjacent pairs

get_stats is the entire "statistics" half of the algorithm. zip(ids, ids[1:]) is the idiomatic 
sliding window of size 2: it pairs each element with its successor and stops naturally at the end (no index 
arithmetic, no off-by-one).

counts.get(pair, 0) + 1 is a hand-rolled defaultdict(int) kept explicit here because the whole point of the lecture
is that nothing is hidden.

The `defaultdict(int)` is hand-rolled on purpose: the point of the lecture is that nothing is hidden.

In [5]:
def get_stats(ids):
    """
    Count how often each adjacent pair appears.

    Args:
        ids: list[int] -- the current token sequence.
    Returns:
        dict[(int, int) -> int] -- pair counts. Insertion order is first-occurence
        order, which quietly determines tie-breaking downstream (see below).
    """
    counts = {}
    for pair in zip(ids, ids[1:]):  # Pythonic way to iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(tokens)
# The most informative view: pairs sorted by frequency, descending.
sorted(((v, k) for k, v in stats.items()), reverse=True)[:10]

[(20, (101, 32)),
 (15, (240, 159)),
 (12, (226, 128)),
 (12, (105, 110)),
 (10, (115, 32)),
 (10, (97, 110)),
 (10, (32, 97)),
 (9, (32, 116)),
 (8, (116, 104)),
 (7, (159, 135))]

In [6]:
# max(stats, key=stats.get) -> the KEY with the largest value.
# Ties are broken by dict insertion order, i.e. whichever pair appeared first in the
# text wins. It is arbitrary but deterministic, which is all we need for reproducibility.
top_pair = max(stats, key=stats.get)
top_pair    # (101, 32) == (ord('e'), ord(' ')) - the trailing "e " of English words

(101, 32)

### Apply a merge

`merge` is the "rewrite" half. It scans left to right and greedily replaces non-overlapping occurrences
of pair with the new id.

**Gotcha — overlapping pairs.** For ids = [1, 1, 1] and pair = (1,1), the scan consumes positions 0-1 and returns 
[256, 1], not [256, 256] (which would be nonsense) and not [1, 256]. Greedy left-to-right is a choice, and both 
training and inference must make the same one, or encode/decode stop agreeing. 

The guard `i < len(ids) - 1` exists so `ids[i+1]` never runs off the end. Drop it and the last element of every sequence
raises IndexError.

In [7]:
def merge(ids, pair, idx):
    """
    Replace every non-overlapping occurence of 'pair' in 'ids' with the single id 'idx'.

    Args:
        ids:  list[int] -- current sequence.
        pair: (int, int) == the two ids to fuse.
        idx:  int -- the new id standing for that pair.
    Returns:
        a NEW list (the input is not mutated).
    """
    # in the list of ints (ids), replace all consecutive occurences of pair with the new token idx
    newids = []
    i = 0
    while i < len(ids):
        # if we are not at the very last position AND the pair matches, replace it
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2  # consume BOTH elements. This is what makes matches non-overlapping. 
        else:
            newids.append(ids[i])
            i += 1
    return newids

# Sanity check on a toy sequence: the (6, 7) inside [5, 6, 6, 7, 9, 1] becomes 99.
# Note the first 6 is left alone. Only the pair that actually matches is rewritten.
print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

tokens2 = merge(tokens, top_pair, 256)
print(tokens2)
print("length:", len(tokens2))  # exactly len(tokens) - count(top_pair)

[5, 6, 99, 9, 1]
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140, 240, 159, 135, 169, 226, 128, 140, 240, 159, 135, 170, 33, 32, 240, 159, 152, 132, 32, 84, 104, 256, 118, 101, 114, 121, 32, 110, 97, 109, 256, 115, 116, 114, 105, 107, 101, 115, 32, 102, 101, 97, 114, 32, 97, 110, 100, 32, 97, 119, 256, 105, 110, 116, 111, 32, 116, 104, 256, 104, 101, 97, 114, 116, 115, 32, 111, 102, 32, 112, 114, 111, 103, 114, 97, 109, 109, 101, 114, 115, 32, 119, 111, 114, 108, 100, 119, 105, 100, 101, 46, 32, 87, 256, 97, 108, 108, 32, 107, 110, 111, 119, 32, 119, 256, 111, 117, 103, 104, 116, 32, 116, 111, 

## The training loop

One merge is a parlour trick; the loop is the algorithm. Two changes from the cells above:

- We switch to a much longer training text — the full blog post, loaded from `data/`. Pair statistics from a single paragraph are noise; BPE is a frequency method and it only says something meaningful about the corpus it saw. The tokenizer's training set is a real design decision: its mix of languages and code determines which strings become cheap.
- We keep `get_stats` and `merge` from above. The self-contained version of this loop is `BasicTokenizer.train` in `tokenizer/basic.py`.

In [8]:
# A ~24 kB article gives pair statistics that mean something. Tokenizer training
# corpora are usually many GB, and their composition (English vs. code vs. other
# languages) is what makes a tokenizer good or bad at each of those.
# text from https://www.reedbeta.com/blog/programmers-intro-to-unicode/
text = open("../data/programmers_intro_to_unicode.txt", encoding="utf-8").read()
tokens = list(text.encode("utf-8"))   # str -> bytes -> list[int] in 0..255
print(len(text), "code points ->", len(tokens), "bytes")

23328 code points -> 24597 bytes


In [9]:
# THE TRAINING LOOP
# This is the whole "training" of a BPE tokenizer. No gradients, no GPU: it is a
# greedy, deterministic counting procedure over the corpus.
vocab_size = 276            # the desired final vocabulary size (256 bytes + 20 learned merges)
num_merges = vocab_size - 256
ids = list(tokens)          # copy so we don't destroy the original list

merges = {} # (int, int) -> int
for i in range(num_merges):
    stats = get_stats(ids)              # recount from scratch: the previous merge
                                        # created new adjacencies that didn't exist before
    pair = max(stats, key=stats.get)    # greedy: most frequent pair wins, ties -> first seen
    idx = 256 + i                       # new ids are handed out in order, starting after the bytes
    print(f"merging {pair} into a new token {idx}")
    ids = merge(ids, pair, idx)         # rewrite the corpus with the merge applied
    merges[pair] = idx                  # records the rule. INSERTION ORDER IS THE MERGE ORDER,
                                        # and we rely on that ordering in both encode() and decode()

merging (101, 32) into a new token 256
merging (105, 110) into a new token 257
merging (115, 32) into a new token 258
merging (116, 104) into a new token 259
merging (101, 114) into a new token 260
merging (99, 111) into a new token 261
merging (116, 32) into a new token 262
merging (226, 128) into a new token 263
merging (44, 32) into a new token 264
merging (97, 110) into a new token 265
merging (111, 114) into a new token 266
merging (100, 32) into a new token 267
merging (97, 114) into a new token 268
merging (101, 110) into a new token 269
merging (257, 103) into a new token 270
merging (261, 100) into a new token 271
merging (121, 32) into a new token 272
merging (46, 32) into a new token 273
merging (97, 108) into a new token 274
merging (259, 256) into a new token 275


Read the printed trace: the tokenizer discovers `(101, 32)` = `"e "`, then `(105, 110)` = `"in"`, 
`(115, 32)` = `"s "`, `(116, 104)` = `"th"`… It is rediscovering English morphology — suffixes, common bigrams, 
the space-before-word convention — purely from counts.

Watch for merges whose *children are themselves merges*: `(257, 103) -> 270` is `"in" + "g" = "ing"`, 
and `(259, 256) -> 275` is `"th" + "e " = "the "`. Tokens form a tree over bytes, which is exactly why the decoding 
table can be built by replaying `merges` in order.

In [10]:
# Compression ratio = bytes per token. ~1.27X here because we only did 20 merges on a
# small corpus. Production tokenizers do ~50k-100k merges and land around 3-4 bytes/token
# on English. That factor is a direct multiplier on how much text fits in the context window.
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length: 24597
ids length: 19438
compression ratio: 1.27X


Note, the Tokenizer is a completely separate, independent module from the LLM. It has its own training dataset 
of text (which could be different from that of the LLM), on which you train the vocabulary using the Byte Pair 
Encoding (BPE) algorithm. It then translates back and forth between raw text and sequences of tokens. 
The LLM later only ever sees the tokens and never directly deals with any text.

The tokenizer is trained on its own corpus, with its own objective (compression), and then frozen. The LLM downstream 
only ever sees integers. Nothing propagates back through the tokenizer. There is no gradient, and there is no way for 
the model to tell you that your tokenization was a bad idea.

## Decoding: ids → text

Given a sequence of integers in the range [0, vocab_size], what is the text?

The decoding table is derived, not stored separately: start with the 256 byte tokens, then replay every merge in 
learned order, concatenating the byte strings of its two children. This works precisely because a merge's children 
always have smaller ids than the merge itself, so by the time we reach idx, vocab[p0] and vocab[p1] are already built.
This relies on dict preserving insertion order. **Gotcha:** `errors="replace"` is not optional — see the last line of the cell.

In [11]:
# Build the decoding table: id -> bytes.
vocab = {idx: bytes([idx]) for idx in range(256)}   # the 256 byte tokens, always present
for (p0, p1), idx in merges.items():                # replay merges IN ORDER
    vocab[idx] = vocab[p0] + vocab[p1]              # a merged token is just its children concatenated

def decode(ids):
    # given ids (list of integers), return Python string
    tokens = b"".join(vocab[idx] for idx in ids)    # ids -> bytes
    text = tokens.decode("utf-8", errors="replace") # bytes -> str; invalid sequences become U+FFFD
    return text

print(decode([128]))    # a lone 0x80 continuation byte -> the replacement character

�


## Encoding: text → ids

The other way around: Given a string, what are the tokens?

Encoding is subtler than it looks. 

The rule: apply merges in the order they were learned. Not in order of frequency in this string, not the longest 
match first, specifically learned order. If the training run learned ("t", "h") -> 259 before ("th", "e ") -> 275, 
then any encoder that builds "the " without first building "th" has produced a token sequence the model was never
trained on. 

So the loop is: find every pair currently present, ask merges which of them has the lowest index (= was learned 
earliest), apply that merge, repeat. This replays the training procedure on a new string. 

Reading min(stats, key=lambda p: merges.get(p, float("inf"))):
    - stats is only used for its keys (the set of pairs present). The counts are irrelevant here, which is a nice
    illustration of why "most frequent pair" is a training-time idea, not an inference-time one.
    - merges.get(p, inf) scores each present pair by when it was learned; pairs that were never learned score inf.
    - If the winner scores inf, no present pair is mergeable and we're done (if pair not in merges: break).
    - The while len(tokens) >= 2 guard handles the empty and single-token cases: with fewer than two ids there are 
    no pairs, and min() over an empty stats would raise ValueError. encode("") must return [] rather than crash.

In [12]:
# The trained model, in its entirety: 20 rules, each mapping a pair to a new id. 
# The ORDER of this dict is the merge order, and it is load-bearing. 
merges

{(101, 32): 256,
 (105, 110): 257,
 (115, 32): 258,
 (116, 104): 259,
 (101, 114): 260,
 (99, 111): 261,
 (116, 32): 262,
 (226, 128): 263,
 (44, 32): 264,
 (97, 110): 265,
 (111, 114): 266,
 (100, 32): 267,
 (97, 114): 268,
 (101, 110): 269,
 (257, 103): 270,
 (261, 100): 271,
 (121, 32): 272,
 (46, 32): 273,
 (97, 108): 274,
 (259, 256): 275}

In [13]:
def encode(text):
    # given a string, return list of integers (the tokens)
    # always start from raw bytes
    tokens = list(text.encode("utf-8"))
    # <2 ids -> no pairs so nothing to do
    while len(tokens) >= 2:
        # we only need the KEYS here (which pairs exist)
        stats = get_stats(tokens)
        # earliest-learned present pair
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            # nothing else can be merged
            break
        idx = merges[pair]
        # apply exactly one merge, then re-scan
        tokens = merge(tokens, pair, idx)
    return tokens

# edge case: empty string -> [] (guard above saves us)
print(encode(""))

[]


In [14]:
# Round trip on a short string.
print(decode(encode("hello world")))

hello world


In [15]:
# Round trip on the full training text. This direction is guaranteed:
#   decode(encode(s)) == s  for every string s
# because encoding starts from the exact UTF-8 bytes and every merge is reversible.
text2 = decode(encode(text))
print(text2 == text)

True


In [16]:
# The check that actually matters: held-out text the tokenizer never trained on.
# Byte-level fallback means coverage is total -- there is no OOV, ever.
valtext = "Many common characters, including numerals, punctuation, and other symbols, are unified within the standard and are not treated as specific to any given writing system. Unicode encodes thousands of emoji, with the continued development thereof conducted by the Consortium as a part of the standard.[4] Moreover, the widespread adoption of Unicode was in large part responsible for the initial popularization of emoji outside of Japan. Unicode is ultimately capable of encoding more than 1.1 million characters."
valtext2 = decode(encode(valtext))
print(valtext2 == valtext)

True


The round trip only holds in one direction. `decode(encode(s)) == s` always. `encode(decode(ids)) == ids` 
Does not hold in general: `[104, 101]` decodes to `"he"`, which re-encodes to whatever merged token covers `"he"`. 
Token sequences are not canonical, and an id list a model produced may not be an id list the encoder would ever produce.
This is behind a whole family of production bugs: caching keyed on token ids, "token healing", and prompts that end 
mid-token.

## Forced splits: the GPT regex

Naive BPE on raw text will happily learn tokens like "dog.", "dog!", "dog?" - one embedding row per punctuation variant,
all of them redundant with "dog". Worse, common words absorb their neighbors and the same concept gets scattered across
many near duplicate tokens. 

GPT-2's fix is a pre-tokenization pass: split the text with a hand-written regex first, then run BPE inside each chunk 
independently. Merges can never cross a chunk boundary, so the categories letters / numbers / punctuation / whitespace
stay separated by construction.

The pattern, piece by piece:

| alternative | matches | why |
|---|---|---|
| `'s\|'t\|'re\|'ve\|'m\|'ll\|'d` | English contractions | keep `'s` as its own chunk instead of gluing it onto the word |
| ` ?\p{L}+` | optional space + letters | the leading space joins the word — `" world"` is one chunk, so the model learns word-with-space as the unit |
| ` ?\p{N}+` | optional space + digits | numbers never merge with letters |
| ` ?[^\s\p{L}\p{N}]+` | optional space + punctuation/symbols | `"!!!?"` groups together but stays away from words |
| `\s+(?!\S)` | whitespace not followed by non-whitespace | the trailing whitespace of a run, minus its last space… |
| `\s+` | remaining whitespace | …so that last space can attach to the next word via the rules above |

In [17]:
import regex as re
# The GPT-2 pre-tokenization pattern, verbatim from OpenAI's encoder.py.
# Text is split by this FIRST; BPE then runs inside each chunk independently,
# so no merge can ever span two chunks.
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
# Note in the output: " world" keeps its leading space, "123" is separate from the word,
# "'ve" and "'s" are split off, and "!!!?" clumps together as punctuation.
print(re.findall(gpt2pat, "Hello've world123 how's are you!!!?"))

['Hello', "'ve", ' world', '123', ' how', "'s", ' are', ' you', '!!!?']


In [18]:
# Python source is a useful stress test: indentation is pure whitespace, and GPT-2's
# pattern refuses to merge spaces with each other in a useful way.
example = """
for i in range(1, 101):
    if i % 3 == 0 and i % 5 == 0:
        print("FizzBuzz")
    elif i % 3 == 0:
        print("Fizz")
    elif i % 5 == 0:
        print("Buzz")
    else:
        print(i)
"""
# Look at the chunks like '\n   ' followed by ' if': every indent level is spent
# as several tokens. This is precisely why GPT-2 was mediocre at Python.
print(re.findall(gpt2pat, example))

['\n', 'for', ' i', ' in', ' range', '(', '1', ',', ' 101', '):', '\n   ', ' if', ' i', ' %', ' 3', ' ==', ' 0', ' and', ' i', ' %', ' 5', ' ==', ' 0', ':', '\n       ', ' print', '("', 'FizzBuzz', '")', '\n   ', ' elif', ' i', ' %', ' 3', ' ==', ' 0', ':', '\n       ', ' print', '("', 'Fizz', '")', '\n   ', ' elif', ' i', ' %', ' 5', ' ==', ' 0', ':', '\n       ', ' print', '("', 'Buzz', '")', '\n   ', ' else', ':', '\n       ', ' print', '(', 'i', ')', '\n']


### What GPT-4 changed

The meaningful changes:

- (?i:...) — contractions match case-insensitively, fixing the "'T" embarrassment.
- \p{N}{1,3} — numbers are chopped into groups of at most 3 digits, so the vocabulary can't be flooded with arbitrary 
long numerals. (It still doesn't guarantee the alignment of those groups, which is a large part of why LLM arithmetic 
is unreliable.)
- Whitespace handling — \s*[\r\n] and the possessive quantifiers let runs of spaces merge, which is why GPT-4 
is dramatically better at indented code.
- The vocabulary roughly doubled: ~50k → ~100k tokens.

In [19]:
import tiktoken   # OpenAI's tokenizer library: INFERENCE ONLY (it cannot train a new tokenizer)

# GPT-2 (does not merge spaces): 4 leading spaces -> 3 separate space tokens + " hello"
enc = tiktoken.get_encoding("gpt2")
print(enc.encode("    hello world!!!"))

# GPT-4 (merges spaces): the same 4 spaces collapse into ONE token
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode("    hello world!!!"))

[220, 220, 220, 23748, 995, 10185]


[262, 24748, 1917, 12340]


In [20]:
# --- [added] Quantify the difference on real code ------------------------------
# Same string, two tokenizers. The ratio is how much more context GPT-4 gets for free
# on Python source -- almost entirely from better whitespace handling.
gpt2_enc = tiktoken.get_encoding("gpt2")
gpt4_enc = tiktoken.get_encoding("cl100k_base")
for name, s in [("python", example), ("english", valtext), ("korean", "안녕하세요 반갑습니다")]:
    a, b = len(gpt2_enc.encode(s)), len(gpt4_enc.encode(s))
    print(f"{name:8s} chars={len(s):5d}  gpt2={a:4d} tokens  cl100k={b:4d} tokens  ({a/b:.2f}x)")

python   chars=  198  gpt2= 109 tokens  cl100k=  72 tokens  (1.51x)
english  chars=  510  gpt2=  94 tokens  cl100k=  94 tokens  (1.00x)
korean   chars=   11  gpt2=  27 tokens  cl100k=   9 tokens  (3.00x)


## What GPT-2 actually shipped

Reference: the GPT-2 [`encoder.py`](https://github.com/openai/gpt-2/blob/master/src/encoder.py). The tokenizer was released as two files, `vocab.bpe` and `encoder.json`; `scripts/download_gpt2.py` fetches them into `assets/gpt2/` (git-ignored) and skips if they are already there.

In [21]:
!python ../scripts/download_gpt2.py

downloading vocab.bpe -> /home/diego-magana/dev/tokenizer/assets/gpt2/vocab.bpe


downloading encoder.json -> /home/diego-magana/dev/tokenizer/assets/gpt2/encoder.json


In [22]:
import json

with open('../assets/gpt2/encoder.json', 'r') as f:
    encoder = json.load(f) # <--- ~equivalent to our "vocab"
    # maps token STRING -> id. The strings look mangled ("Ġhello") because GPT-2 passes
    # every byte through a reversible byte<->unicode map (the `bytes_to_unicode` function
    # in encoder.py) so that no token contains a control character or a raw space.
    # "Ġ" is just byte 0x20 (space) wearing a disguise. Cosmetic complexity, no new ideas.

with open('../assets/gpt2/vocab.bpe', 'r', encoding="utf-8") as f:
    bpe_data = f.read()
# One merge per line, IN LEARNED ORDER -- line 0 is a version header, last line is empty.
bpe_merges = [tuple(merge_str.split()) for merge_str in bpe_data.split('\n')[1:-1]]
# ^---- ~equivalent to our "merges"

So GPT-2's tokenizer is the same two objects we built — merges (ordered pairs) and vocab (id → bytes) — 
plus one cosmetic byte-shuffling layer. That's the whole thing. Everything else in `encoder.py` is bookkeeping.

In [23]:
# --- [added] Confirm the correspondence with our own two dicts -------------------
print("number of merges :", len(bpe_merges))   # 50,000
print("first 5 merges   :", bpe_merges[:5])    # exactly the same shape as our `merges` keys
print("vocab entries    :", len(encoder))      # 256 bytes + 50,000 merges + 1 special = 50,257

number of merges : 50000
first 5 merges   : [('Ġ', 't'), ('Ġ', 'a'), ('h', 'e'), ('i', 'n'), ('r', 'e')]
vocab entries    : 50257


## Special tokens

Special tokens are ids appended *after* the BPE vocabulary, matched by exact string lookup before BPE ever runs. 
They carry structural meaning rather than text: document boundaries, chat turn boundaries, fill-in-the-middle markers.

GPT-2 has exactly one: <|endoftext|> (id 50256), used to separate documents in the training stream so the model learns 
that context does not flow across the boundary. GPT-3.5/4 added several more to encode chat structure 
(<|im_start|>, <|im_end|>) — the entire "system / user / assistant" abstraction is, at the token level, just a handful 
of special ids.

Adding a special token is model surgery, not a config change. You must grow the embedding matrix by a row and the 
output (unembedding) matrix by a column, initialize them (small random values), and fine-tune so they mean something. 
This is exactly what happens when a base model is turned into a chat model.

In [24]:
len(encoder) # 256 raw byte tokens. 50,000 merges. +1 special token

50257

In [25]:
encoder['<|endoftext|>'] # the only special token in use for the GPT-2 base model

50256

## From here: the minbpe exercise

At this point you have everything you need to build your own GPT-4 tokenizer. This is the [exercise progression]
(https://github.com/karpathy/minbpe/blob/master/exercise.md) you may wish to follow. You'll note that it is part of 
the [minbpe](https://github.com/karpathy/minbpe) repo, which is the solution to that exercise, and is a cleaned up 
version of the code above.

The [exercise progression](https://github.com/karpathy/minbpe/blob/master/exercise.md) in four steps:

1. **`BasicTokenizer`** — `train`, `encode`, `decode` on raw bytes. That's everything above, wrapped in a class: `tokenizer.BasicTokenizer`.
2. **`RegexTokenizer`** — add a split pattern so merges respect chunk boundaries (`tokenizer.RegexTokenizer`, with GPT-2's pattern).
3. **Match `tiktoken`** — load GPT-4's published merges and reproduce `cl100k_base`'s output exactly. The one trap is 
that GPT-4 permutes the 256 byte tokens, so you need its byte shuffle to match ids.
4. **Special tokens** — handle `<|endoftext|>` and friends in `encode`.

In [26]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # GPT-4 tokenizer
print(enc.encode("안녕하세요 👋 (hello in Korean!)"))
print(enc.decode(enc.encode("안녕하세요 👋 (hello in Korean!)")) == "안녕하세요 👋 (hello in Korean!)")
# Target for your own implementation: identical id list, and a lossless round trip.
# Note how many tokens the Korean costs relative to the English -- the tokenizer's
# training mix is visible right here in the token count.
# match the above for your own tokenizer, and also implement a train() function

[31495, 230, 75265, 243, 92245, 62904, 233, 320, 15339, 304, 16526, 16715]
True


## SentencePiece (Llama / Mistral lineage)

Commonly used because, unlike tiktoken, it can efficiently both train and run BPE tokenizers. It is used in the Llama and Mistral series: <https://github.com/google/sentencepiece>.

The one real difference:

- **tiktoken-style (GPT):** encode to UTF-8 first, then run BPE over *bytes*. Every string is representable by construction; there is no UNK token and no concept of an unseen character.
- **SentencePiece:** run BPE over *code points*. Rare code points (governed by `character_coverage`) are either mapped to UNK or, with `byte_fallback=True`, expanded into their UTF-8 bytes and encoded as byte tokens.
- SentencePiece also trains, which tiktoken cannot — that's why it's the default for open-weight models.

In [27]:
import sentencepiece as spm

In [28]:
# write a toy.txt file with some random text
# Docs: https://github.com/google/sentencepiece/blob/master/doc/options.md
#       https://github.com/google/sentencepiece/blob/master/src/sentencepiece_model.proto#L193
with open("toy.txt", "w", encoding="utf-8") as f:
  f.write("SentencePiece is an unsupervised text tokenizer and detokenizer mainly for Neural Network-based text generation systems where the vocabulary size is predetermined prior to the neural model training. SentencePiece implements subword units (e.g., byte-pair-encoding (BPE) [Sennrich et al.]) and unigram language model [Kudo.]) with the extension of direct training from raw sentences. SentencePiece allows us to make a purely end-to-end system that does not depend on language-specific pre/postprocessing.")

In [29]:
# train a sentencepiece model on it
# the settings here are (best effort) those used for training Llama 2
import os

options = dict(
  # --- input spec ---
  input="toy.txt",
  input_format="text",
  # --- output spec ---
  model_prefix="tok400", # output filename prefix
  # --- algorithm spec ---
  # BPE alg  (the alternative, and sentencepiece's default, is "unigram")
  model_type="bpe",
  vocab_size=400,
  # --- normalization ---
  # sentencepiece grew out of machine translation, where normalizing/lowercasing text
  # was normal. For an LLM you want the bytes untouched, hence "identity".
  normalization_rule_name="identity", # ew, turn off normalization
  remove_extra_whitespaces=False,
  input_sentence_size=200000000, # max number of training sentences
  max_sentence_length=4192, # max number of bytes per sentence
  seed_sentencepiece_size=1000000,
  shuffle_input_sentence=True,
  # --- rare word treatment ---
  # any code point rarer than this coverage threshold is NOT given its own token;
  # with byte_fallback it is emitted as raw UTF-8 byte tokens instead of <unk>.
  character_coverage=0.99995,
  byte_fallback=True,
  # --- merge rules (sentencepiece's equivalent of GPT's split regex) ---
  split_digits=True,             # never merge digits together
  split_by_unicode_script=True,  # never merge across scripts (Latin/Hangul/...)
  split_by_whitespace=True,
  split_by_number=True,
  max_sentencepiece_length=16,   # cap on token length in characters
  add_dummy_prefix=True,         # prepend " " to every input so "world" == " world"
  allow_whitespace_only_pieces=True,
  # --- special tokens ---
  unk_id=0, # the UNK token MUST exist
  bos_id=1, # the others are optional, set to -1 to turn off
  eos_id=2,
  pad_id=-1,
  # --- systems ---
  num_threads=os.cpu_count(), # use ~all system resources
)

spm.SentencePieceTrainer.train(**options)

I0000 00:00:1790805584.106839  177348 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: toy.txt
  input_format: text
  model_prefix: tok400
  model_type: BPE
  vocab_size: 400
  self_test_sample_size: 0
  character_coverage: 0.99995
  input_sentence_size: 200000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 12
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 1
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_d

True

In [30]:
sp = spm.SentencePieceProcessor()
sp.load('tok400.model')
# Read the vocabulary top to bottom and you can see the layout:
#   0,1,2    -> <unk>, <s>, </s>        (special tokens come FIRST here, unlike GPT)
#   3..258   -> <0x00>..<0xFF>          (the byte-fallback tokens)
#   then     -> learned merges, then individual characters
# named sp_vocab so it doesn't shadow the BPE `vocab` dict that decode() reads
sp_vocab = [[sp.id_to_piece(idx), idx] for idx in range(sp.get_piece_size())]
sp_vocab

[['<unk>', 0],
 ['<s>', 1],
 ['</s>', 2],
 ['<0x00>', 3],
 ['<0x01>', 4],
 ['<0x02>', 5],
 ['<0x03>', 6],
 ['<0x04>', 7],
 ['<0x05>', 8],
 ['<0x06>', 9],
 ['<0x07>', 10],
 ['<0x08>', 11],
 ['<0x09>', 12],
 ['<0x0A>', 13],
 ['<0x0B>', 14],
 ['<0x0C>', 15],
 ['<0x0D>', 16],
 ['<0x0E>', 17],
 ['<0x0F>', 18],
 ['<0x10>', 19],
 ['<0x11>', 20],
 ['<0x12>', 21],
 ['<0x13>', 22],
 ['<0x14>', 23],
 ['<0x15>', 24],
 ['<0x16>', 25],
 ['<0x17>', 26],
 ['<0x18>', 27],
 ['<0x19>', 28],
 ['<0x1A>', 29],
 ['<0x1B>', 30],
 ['<0x1C>', 31],
 ['<0x1D>', 32],
 ['<0x1E>', 33],
 ['<0x1F>', 34],
 ['<0x20>', 35],
 ['<0x21>', 36],
 ['<0x22>', 37],
 ['<0x23>', 38],
 ['<0x24>', 39],
 ['<0x25>', 40],
 ['<0x26>', 41],
 ['<0x27>', 42],
 ['<0x28>', 43],
 ['<0x29>', 44],
 ['<0x2A>', 45],
 ['<0x2B>', 46],
 ['<0x2C>', 47],
 ['<0x2D>', 48],
 ['<0x2E>', 49],
 ['<0x2F>', 50],
 ['<0x30>', 51],
 ['<0x31>', 52],
 ['<0x32>', 53],
 ['<0x33>', 54],
 ['<0x34>', 55],
 ['<0x35>', 56],
 ['<0x36>', 57],
 ['<0x37>', 58],
 ['<0x38>', 5

In [31]:
# "hello" is in the training text; the Korean is not, and this toy model has only
# 400 tokens, so the Korean falls through to byte tokens.
# named sp_ids so it doesn't shadow the BPE `ids` from the training loop
sp_ids = sp.encode("hello 안녕하세요")
print(sp_ids)

[362, 378, 361, 372, 358, 362, 239, 152, 139, 238, 136, 152, 240, 152, 155, 239, 135, 187, 239, 157, 151]


In [32]:
# The pieces make the fallback visible: '▁' is the dummy-prefix space marker, and
# every '<0x..>' is a raw UTF-8 byte that had no learned token.
print([sp.id_to_piece(idx) for idx in sp_ids])

['▁', 'h', 'e', 'l', 'lo', '▁', '<0xEC>', '<0x95>', '<0x88>', '<0xEB>', '<0x85>', '<0x95>', '<0xED>', '<0x95>', '<0x98>', '<0xEC>', '<0x84>', '<0xB8>', '<0xEC>', '<0x9A>', '<0x94>']


## The same thing as a package

Everything above, as a class with save/load: `tokenizer/basic.py`. It is the same code, not a rewrite — the check below is that the class, trained on the same text to the same vocabulary size, learns byte-for-byte the same merges as the loop in this notebook.

In [33]:
from tokenizer import BasicTokenizer

tok = BasicTokenizer()
tok.train(text, vocab_size=276)

assert tok.merges == merges                      # identical rules, in identical order
assert tok.encode(text) == ids                   # identical ids on the training text
assert tok.decode(tok.encode(valtext)) == valtext
print("package matches the notebook:", len(tok.merges), "merges")

package matches the notebook: 20 merges


In [34]:
import tempfile, os

# save/load round trip — the .model file is the same shape as GPT-2's vocab.bpe
with tempfile.TemporaryDirectory() as d:
    tok.save(os.path.join(d, "walkthrough_276"))
    print(open(os.path.join(d, "walkthrough_276.model")).read()[:120], "...")
    loaded = BasicTokenizer()
    loaded.load(os.path.join(d, "walkthrough_276.model"))
assert loaded.encode(valtext) == tok.encode(valtext)

bpe v1

0
101 32
105 110
115 32
116 104
101 114
99 111
116 32
226 128
44 32
97 110
111 114
100 32
97 114
101 110
257 103 ...
